# Qwen2.5-Omni-3B 오디오 멀티모달 QLoRA 파인튜닝 (Kaggle Notebooks)

API 키 공유 없이 **브라우저에서 직접 실행**하는 노트북입니다 (Colab판과 동일한 파이프라인).

**실행 전 필수 설정** (우측 패널 `Settings`):
1. **Accelerator**: GPU T4 x2 또는 P100 선택
2. **Internet**: On (레포 클론·패키지 설치에 필요)

Kaggle GPU(T4/P100)는 둘 다 bf16을 네이티브 지원하지 않으므로 학습은 자동으로 **fp16**으로 진행됩니다.


## 1. GPU 확인


In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print('GPU:', name, '| compute capability:', cap)
    bf16_ok = cap[0] >= 8
    print('bf16 지원:', bf16_ok, '→ 학습 dtype:', 'bfloat16' if bf16_ok else 'float16')

## 2. 레포 클론
Kaggle 작업 디렉터리(`/kaggle/working`)에 클론합니다.


In [ ]:
%cd /kaggle/working
!git clone --branch claude/local-multimodal-model-8mluhh https://github.com/minsik1313/minsik.git
%cd minsik
!ls -la

## 3. 의존성 설치
Kaggle 기본 이미지에는 torch가 이미 설치되어 있습니다. ms-swift와 오디오/평가 패키지만 추가합니다.


In [ ]:
!pip install -q 'ms-swift[all]' librosa soundfile jiwer qwen-omni-utils
!swift --version

## 4. 데이터 준비 (합성 샘플)
smoke test용 합성 오디오 데이터를 생성합니다. 실데이터는 `scripts/prepare_data.py`의
`build_records_from_real_data()`를 구현한 뒤 `--synthesize` 없이 실행하세요.


In [ ]:
!python scripts/prepare_data.py --synthesize --n-per-class 12
!head -1 data/train.jsonl

## 5. 학습 (QLoRA)
T4/P100 기준 **fp16 + MAX_LENGTH=1024**로 자동 실행됩니다.


In [ ]:
import torch
dtype = 'bfloat16' if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else 'float16'
print('학습 dtype =', dtype)
import os
os.environ['TORCH_DTYPE'] = dtype
os.environ['COMPUTE_DTYPE'] = dtype
os.environ['MAX_LENGTH'] = '1024'
os.environ['EPOCHS'] = '1'
!TORCH_DTYPE=$TORCH_DTYPE COMPUTE_DTYPE=$COMPUTE_DTYPE MAX_LENGTH=$MAX_LENGTH EPOCHS=$EPOCHS bash scripts/train.sh

## 6. 추론


In [ ]:
!bash scripts/infer.sh

## 7. 평가


In [ ]:
!python scripts/eval.py --metric accuracy

## 8. 결과 보존
Kaggle 세션이 종료되면 `/kaggle/working`의 파일이 사라질 수 있습니다.
우측 상단 **Save Version**(Save & Run All)을 실행하면 `outputs/`가 Kaggle Notebook Output으로 영구 저장되어
이후 `Output` 탭에서 다운로드할 수 있습니다.


In [ ]:
!zip -r outputs.zip outputs
print('outputs.zip 생성 완료 — Save Version 실행 시 Output 탭에서 다운로드 가능')